In [ ]:
import torch
import torch.nn as nn

torch.manual_seed(42)
print(f"PyTorch {torch.__version__} | Device: cpu")

In [ ]:
GPT_CONFIG_124M = {
    "vocab_size": 50257,             # Vocabulary size
    "max_position_embeddings": 1024, # Context length
    "hidden_size": 768,              # Embedding dimension
    "num_attention_heads": 12,       # Number of attention heads
    "num_hidden_layers": 12,         # Number of layers
    "attention_dropout": 0.1,        # Dropout rate
    "attention_bias": False          # Query-Key-Value bias
}

In [ ]:
import tiktoken

tokenizer = tiktoken.get_encoding("gpt2")

start_context = "Hi my name is"

encoded = tokenizer.encode(start_context)
input_ids = torch.tensor(encoded).unsqueeze(0)

In [ ]:
class CausalAttention(nn.Module):

    def __init__(self, hidden_size, max_position_embeddings,
                 attention_dropout, attention_bias=False):
        super().__init__()
        self.hidden_size = hidden_size
        self.q_proj = nn.Linear(hidden_size, hidden_size, bias=attention_bias)
        self.k_proj = nn.Linear(hidden_size, hidden_size, bias=attention_bias)
        self.v_proj = nn.Linear(hidden_size, hidden_size, bias=attention_bias)
        self.dropout = nn.Dropout(attention_dropout) # New
        self.register_buffer('causal_mask', torch.triu(torch.ones(max_position_embeddings, max_position_embeddings), diagonal=1)) # New

    def forward(self, x):
        batch_size, seq_len, _ = x.shape
        key_states = self.k_proj(x)
        query_states = self.q_proj(x)
        value_states = self.v_proj(x)

        # attn_scores = query_states @ key_states.T # THIS IS WRONG so we will comment this
        attn_scores = query_states @ key_states.transpose(1, 2) 
        attn_scores.masked_fill_(  
            self.causal_mask.bool()[:seq_len, :seq_len], -torch.inf)  
        attn_weights = torch.softmax(
            attn_scores / key_states.shape[-1]**0.5, dim=-1
        )
        attn_weights = self.dropout(attn_weights) # New

        attn_output = attn_weights @ value_states
        return attn_output


In [ ]:
class MultiheadAttention(nn.Module):
    def __init__(self, hidden_size, num_heads, max_position_embeddings,
                 attention_dropout, attention_bias=False):
        super().__init__()
        assert hidden_size % num_heads == 0, "Hidden size must be divisible by the number of heads"

        self.num_heads = num_heads
        self.head_dim = hidden_size // num_heads

        # One projection each for queries, keys and values.
        self.q_proj = nn.Linear(hidden_size, hidden_size, bias=attention_bias)
        self.k_proj = nn.Linear(hidden_size, hidden_size, bias=attention_bias)
        self.v_proj = nn.Linear(hidden_size, hidden_size, bias=attention_bias)

        # Mixes the heads back together after they are concatenated.
        self.o_proj = nn.Linear(hidden_size, hidden_size)

        self.dropout = nn.Dropout(attention_dropout)

        # Upper-triangular = the positions each token is NOT allowed to see.
        self.register_buffer(
            "causal_mask",
            torch.triu(torch.ones(max_position_embeddings, max_position_embeddings), diagonal=1).bool(),
        )

    def forward(self, x):
        batch_size, seq_len, _ = x.shape

        query_states = self.q_proj(x)
        key_states = self.k_proj(x)
        value_states = self.v_proj(x)

        # Split the embedding across heads: (B, T, C) -> (B, H, T, head_dim)
        query_states = query_states.view(batch_size, seq_len, self.num_heads, self.head_dim).transpose(1, 2)
        key_states = key_states.view(batch_size, seq_len, self.num_heads, self.head_dim).transpose(1, 2)
        value_states = value_states.view(batch_size, seq_len, self.num_heads, self.head_dim).transpose(1, 2)

        # How much each token cares about every other token, scaled by sqrt(head_dim).
        attn_scores = query_states @ key_states.transpose(-2, -1) / self.head_dim**0.5   # (B, H, T, T)

        # No peeking ahead: future positions become -inf, so softmax sends them to 0.
        attn_scores = attn_scores.masked_fill(
            self.causal_mask[:seq_len, :seq_len], float("-inf")
        )

        attn_weights = torch.softmax(attn_scores, dim=-1)       # (B, H, T, T)
        attn_weights = self.dropout(attn_weights)

        # Weighted average of the values.
        attn_output = attn_weights @ value_states               # (B, H, T, head_dim)

        # Concatenate the heads back into one vector per token.
        attn_output = attn_output.transpose(1, 2).contiguous().view(batch_size, seq_len, -1)

        return self.o_proj(attn_output)

In [ ]:
class GPTModel(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.embed_tokens = nn.Embedding(cfg["vocab_size"], cfg["hidden_size"])
        self.embed_positions = nn.Embedding(cfg["max_position_embeddings"], cfg["hidden_size"])
        # self.drop_emb = nn.Dropout(cfg["attention_dropout"])

        self.layers = nn.ModuleList()
        for _ in range(cfg["num_hidden_layers"]):
            self.layers.append(nn.ModuleDict({
                # Layer norm before attention 
                "input_layernorm": nn.LayerNorm(cfg["hidden_size"]),
                # Multi-head self-attention 
                "self_attn": MultiheadAttention(
                    hidden_size=cfg["hidden_size"],
                    num_heads=cfg["num_attention_heads"],
                    max_position_embeddings=cfg["max_position_embeddings"],
                    attention_dropout=cfg["attention_dropout"],
                    attention_bias=cfg["attention_bias"],
                ),
                # Layer norm before MLP
                "post_attention_layernorm": nn.LayerNorm(cfg["hidden_size"]),
                # Feed-forward MLP: cfg["hidden_size"] -> 4x cfg["hidden_size"] -> cfg["hidden_size"]
                "mlp": nn.Sequential(
                    nn.Linear(cfg["hidden_size"], 4 * cfg["hidden_size"]),
                    nn.GELU(),
                    nn.Linear(4 * cfg["hidden_size"], cfg["hidden_size"]),
                ),
            }))

        self.norm = nn.LayerNorm(cfg["hidden_size"])
        self.lm_head = nn.Linear(cfg["hidden_size"], cfg["vocab_size"], bias=False)

    def forward(self, input_ids):
        batch_size, seq_len = input_ids.shape
        inputs_embeds = self.embed_tokens(input_ids)
        position_embeds = self.embed_positions(torch.arange(seq_len, device=input_ids.device))
        x = inputs_embeds + position_embeds  # Shape [batch_size, seq_len, hidden_size]
        # x = self.drop_emb(x)

        for layer in self.layers:
            # Pre-norm + self-attention + residual connection
            normed = layer["input_layernorm"](x)
            x = x + layer["self_attn"](normed)  # Residual connection

            # Pre-norm + MLP + residual connection
            normed = layer["post_attention_layernorm"](x)
            x = x + layer["mlp"](normed)  # Residual connection

        x = self.norm(x)
        logits = self.lm_head(x)
        return logits

In [ ]:
model = GPTModel(GPT_CONFIG_124M)

In [ ]:
def generate(model, input_ids, max_new_tokens, max_seq_len):
    """Generate tokens one at a time WITHOUT a KV cache.
    Each step re-processes the full sequence (intentionally naive)."""

    for _ in range(max_new_tokens):
        # Crop to the context the model supports. E.g. if the model supports
        # 5 tokens and we already have 10, only the last 5 are fed in.
        context_ids = input_ids[:, -max_seq_len:]

        logits = model(context_ids)          # (B, T, vocab_size)

        # Only the LAST position predicts the next token.
        next_token_logits = logits[:, -1, :]                     # (B, vocab_size)
        probs = torch.softmax(next_token_logits, dim=-1)         # (B, vocab_size)
        next_token = torch.argmax(probs, dim=-1, keepdim=True)   # (B, 1)

        input_ids = torch.cat([input_ids, next_token], dim=1)

    return input_ids

In [ ]:
model.eval()  # disable dropout

out = generate(
    model=model,
    input_ids=input_ids,
    max_new_tokens=6,
    max_seq_len=GPT_CONFIG_124M["max_position_embeddings"],
)

print(tokenizer.decode(out[0].tolist()))